<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex07.2-heat-and-wave/Ex07.2_01_die_soft_ic_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_07.2 · Notebook 01 — The Die, with a **Soft** Initial Condition

**Paired with L7.2 · Fundamental PDEs**

Three loss terms now, not two. The PDE lives in the slab, the edges live on a
tube through time, and the initial field lives on one slice at $t = 0$:

$$\mathcal{L} = \mathcal{L}_{\mathrm{PDE}}
+ w_b\,\mathcal{L}_{\mathrm{edges}} + w_0\,\mathcal{L}_{\mathrm{IC}}$$

All three soft. All three fighting each other.

## What you will do

1. Write the parabolic residual and the three-term loss.
2. Train, then look at *where in time* the error is — not just how big it is.
3. Discover that the initial slice is the hardest part, and work out why.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex07.2-heat-and-wave/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · The residual

$\mathcal{F} = \hat\theta_{,t} - \alpha(\hat\theta_{,xx} + \hat\theta_{,yy})$.

Time is column 2, so `grad(theta, xyt)[:, 2:3]` is $\hat\theta_{,t}$.

### Your turn

In [ ]:
# TODO 1 --- the parabolic residual ----------------------------------------------------------
# Two `...` to replace:
#   line 1  ->  grad(theta, xyt)[:, 2:3]              theta_t: column 2 of the gradient is time
#   line 2  ->  theta_t - pb.ALPHA * lap              theta_t = alpha (theta_xx + theta_yy)
# Only ONE time derivative: first order in time, so one initial condition determines it.
def heat_residual(model, xyt):
    theta = model(xyt)
    theta_t = ...                                 # <- grad(theta, xyt)[:, 2:3]
    lap = d2(theta, xyt, 0) + d2(theta, xyt, 1)
    return ...                                    # <- theta_t - pb.ALPHA * lap
# ------------------------------------------------------------------------------

## 2 · The three-term loss, non-dimensionalised

Ex_07.1 divided the physics residual by the peak source. The same argument
applies here and the scale is different: the residual is a rate, in K/s, and
the natural scale is the peak amplitude times the fastest rate.

### Your turn

In [ ]:
# TODO 2 --- the three-term loss ---------------------------------------------------------------
# Two `...` to replace:
#   line 1  ->  (model(xyt_0) - theta_0) / T_SCALE               the initial field, made dimensionless
#   line 2  ->  mse(f) + w_b * mse(b) + w_0 * mse(i0)             physics, edges, initial condition
k_slow, k_fast = pb.heat_rates()
R_SCALE = (pb.A_SLOW + pb.A_FAST) * k_fast          # K/s
T_SCALE = pb.A_SLOW + pb.A_FAST                     # K

def make_loss(model, xyt_f, xyt_b, xyt_0, theta_0, w_b=1.0, w_0=1.0):
    def loss():
        f  = heat_residual(model, xyt_f) / R_SCALE
        b  = model(xyt_b) / T_SCALE                 # edges are at zero
        i0 = ...                                    # <- (model(xyt_0) - theta_0) / T_SCALE
        return ...                                  # <- mse(f) + w_b * mse(b) + w_0 * mse(i0)
    return loss
# ------------------------------------------------------------------------------

## 3 · Train

In [ ]:
N_F, N_B, N_0 = 4000, 25, 600

set_seed(88)
model = MLP(n_in=3, n_hidden=40, n_layers=4)
describe(model, N_F)

xyt_f = to_tensor(spacetime_points(N_F, pb.HEAT_DOMAIN, (0.0, pb.HEAT_T_END), seed=1),
                  requires_grad=True)
xyt_b = to_tensor(boundary_points_in_time(N_B, 20, pb.HEAT_DOMAIN,
                                          (0.0, pb.HEAT_T_END), seed=1))
init_np = initial_points(N_0, pb.HEAT_DOMAIN, t0=0.0, seed=1)
xyt_0 = to_tensor(init_np)
theta_0 = to_tensor(pb.heat_initial(init_np[:, 0], init_np[:, 1]).reshape(-1, 1))

history = train_two_stage(model, make_loss(model, xyt_f, xyt_b, xyt_0, theta_0),
                          adam_steps=4000, lbfgs_steps=200, lr=1e-3)
plot_curves(history, title="the die — three soft terms")
plt.show()

## 4 · Where in time is the error?

The usual report is one number for the whole slab. That hides the thing worth
knowing: a transient problem is not uniformly hard, and a PINN is usually worst
exactly where the solution is most structured.

### Your turn

In [ ]:
# TODO 3 --- score the model instant by instant --------------------------------------------------
# Two `...` to replace, inside the loop:
#   line 1  ->  np.concatenate([pts, np.full((len(pts), 1), t)], axis=1)     the grid at time t, (N, 3)
#   line 2  ->  pb.heat_exact(pts[:, 0], pts[:, 1], t)                       the exact field at t
TIMES = [0.0, 0.005, 0.02, 0.05, 0.10, 0.20]
X, Y, pts = grid_points(121, 121, pb.HEAT_DOMAIN)

when = {"rel": [], "max": []}
for t in TIMES:
    q = ...                                       # <- np.concatenate([pts, np.full((len(pts), 1), t)], axis=1)
    with torch.no_grad():
        pred = to_numpy(model(to_tensor(q))).ravel()
    ref = ...                                     # <- pb.heat_exact(pts[:, 0], pts[:, 1], t)
    when["rel"].append(relative_l2(pred, ref))
    when["max"].append(max_abs_error(pred, ref))
# ------------------------------------------------------------------------------

In [ ]:
print(error_table(
    [[f"{t*1e3:.0f}", f"{r:.3e}", f"{m:.3f}"]
     for t, r, m in zip(TIMES, when["rel"], when["max"])],
    ["t [ms]", "relative L2", "worst error [K]"]))

fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.2))
axes[0].semilogy(np.array(TIMES)*1e3, when["rel"], "o-", lw=1.9, ms=6,
                 color="#1f77b4")
axes[0].set_xlabel("t [ms]"); axes[0].set_ylabel("relative L2")
axes[0].set_title("Error against time"); axes[0].grid(alpha=0.25, which="both")
axes[1].plot(np.array(TIMES)*1e3, when["max"], "s-", lw=1.9, ms=6,
             color="#d94f2b")
axes[1].set_xlabel("t [ms]"); axes[1].set_ylabel("worst error [K]")
axes[1].set_title("...and in kelvin"); axes[1].grid(alpha=0.25)
plt.show()

**What you should see.** The relative error **largest at $t = 0$** and falling
as the field simplifies.

Three reasons, and it is worth separating them:

1. **The initial slice is the most structured field in the problem.** It
   carries both modes at full amplitude. By 100 ms the sharp one is gone and
   the network is fitting a single smooth lobe.
2. **The initial condition is one term among three.** The optimiser is free to
   trade accuracy at $t = 0$ for a smaller residual in the bulk — and the bulk
   is where almost all the collocation points are.
3. **$t = 0$ is a boundary of the slab.** The network has data on one side of
   it only, exactly as at a spatial edge.

That first point deserves a moment. **The initial condition is the only place
in this problem where you were given the answer**, and soft enforcement spent
some of it.

---

## 5 · What the model actually produced

In [ ]:
show = [0.0, 0.02, 0.10]
fig, axes = plt.subplots(2, len(show), figsize=(13.0, 7.0))
X, Y, pts = grid_points(121, 121, pb.HEAT_DOMAIN)
for j, t in enumerate(show):
    q = np.concatenate([pts, np.full((len(pts), 1), t)], axis=1)
    with torch.no_grad():
        pred = to_numpy(model(to_tensor(q))).ravel()
    ref = pb.heat_exact(pts[:, 0], pts[:, 1], t)
    pb.plot_slice(pred, pb.HEAT_DOMAIN, ax=axes[0, j],
                  title=f"model, t = {t*1e3:.0f} ms", label="θ [K]", scale=1e3)
    pb.plot_slice(pred - ref, pb.HEAT_DOMAIN, ax=axes[1, j],
                  title=f"error, t = {t*1e3:.0f} ms", label="K",
                  cmap="coolwarm", scale=1e3)
plt.tight_layout(); plt.show()

## 6 · Save

In [ ]:
os.makedirs("Ex07.2_outputs", exist_ok=True)
path = os.path.join("Ex07.2_outputs", "nb01_die_soft.npz")
np.savez(path, times=np.asarray(TIMES),
         rel=np.asarray(when["rel"], dtype=float),
         max_err=np.asarray(when["max"], dtype=float),
         adam=history["adam"], lbfgs=history["lbfgs"])
torch.save(model.state_dict(), os.path.join("Ex07.2_outputs", "nb01_die_soft.pt"))
print("wrote", path)

## 7 · Before you move on

1. The error was largest at $t = 0$. Give the three reasons in your own words,
   and say which you think dominates here.
2. Both the edge term and the initial term were divided by the same
   temperature scale. Why does that matter, and what would `w_0 = 1` have meant
   without it?
3. Almost all the collocation points sit in the bulk of the slab. What does
   that do to the balance between the three terms, and how would you change it?

Next: **notebook 02**, where the initial condition is built in and cannot be
traded away.